In [0]:
# =========================================================
# GOLD LAYER — AGGREGATION VIEWS (TOP 3 CATEGORIES)
# =========================================================
# Category 1: Traffic Flow (5 views)
# Category 2: Incidents    (5 views)
# Category 3: Speed        (4 views)
# =========================================================
# All views are created in traffic_catalog.gold schema
# Built on fact_traffic_events joined to dimension tables
# =========================================================

import builtins
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS traffic_catalog.gold")

views_created = []

# =========================================================
# CATEGORY 1 — TRAFFIC FLOW
# =========================================================

print("=" * 50)
print("CATEGORY 1 — TRAFFIC FLOW")
print("=" * 50)

# ── View 1: hourly_congestion_summary ────────────────────
# Answers: Which sensor/location is most congested by hour?
# Business use: identify bottlenecks, plan signal timing

spark.sql("""
CREATE OR REPLACE VIEW traffic_catalog.gold.vw_hourly_congestion_summary AS
SELECT
    f.sensor_id,
    ds.location_id,
    ds.lane_id,
    dt.hour                                         AS event_hour,
    dt.time_of_day,
    dt.is_peak_hour,
    dd.full_date                                    AS event_date,
    dd.weekday_name,
    dd.is_weekend,

    -- Congestion metrics
    ROUND(AVG(f.congestion_score), 2)               AS avg_congestion_score,
    ROUND(MAX(f.congestion_score), 2)               AS max_congestion_score,
    ROUND(MIN(f.congestion_score), 2)               AS min_congestion_score,

    -- Speed metrics
    ROUND(AVG(f.avg_speed), 2)                      AS avg_speed_kmh,

    -- Volume metrics
    SUM(f.vehicle_count)                            AS total_vehicles,
    ROUND(AVG(f.occupancy_rate), 4)                 AS avg_occupancy_rate,
    ROUND(AVG(f.traffic_density), 2)                AS avg_traffic_density,

    -- Congestion level distribution
    COUNT(CASE WHEN f.congestion_score < 25  THEN 1 END) AS low_congestion_hours,
    COUNT(CASE WHEN f.congestion_score < 50
               AND f.congestion_score >= 25 THEN 1 END) AS medium_congestion_hours,
    COUNT(CASE WHEN f.congestion_score < 75
               AND f.congestion_score >= 50 THEN 1 END) AS high_congestion_hours,
    COUNT(CASE WHEN f.congestion_score >= 75 THEN 1 END) AS critical_congestion_hours,
    COUNT(*)                                         AS total_readings

FROM traffic_catalog.gold.fact_traffic_events     f
JOIN traffic_catalog.gold.dim_sensor              ds ON f.sensor_id  = ds.sensor_id
JOIN traffic_catalog.gold.dim_time                dt ON f.time_key   = dt.time_key
JOIN traffic_catalog.gold.dim_date                dd ON f.date_key   = dd.date_key

GROUP BY
    f.sensor_id, ds.location_id, ds.lane_id,
    dt.hour, dt.time_of_day, dt.is_peak_hour,
    dd.full_date, dd.weekday_name, dd.is_weekend
""")
views_created.append("vw_hourly_congestion_summary")
print("  ✓ vw_hourly_congestion_summary")

# ── View 2: peak_vs_offpeak_analysis ─────────────────────
# Answers: How much worse is congestion during peak hours?
# Business use: justify investment in peak-hour interventions

spark.sql("""
CREATE OR REPLACE VIEW traffic_catalog.gold.vw_peak_vs_offpeak_analysis AS
SELECT
    f.sensor_id,
    ds.location_id,
    dt.is_peak_hour,
    dt.time_of_day,

    ROUND(AVG(f.congestion_score), 2)               AS avg_congestion_score,
    ROUND(AVG(f.avg_speed), 2)                      AS avg_speed_kmh,
    ROUND(AVG(f.occupancy_rate), 4)                 AS avg_occupancy_rate,
    SUM(f.vehicle_count)                            AS total_vehicles,
    ROUND(AVG(f.traffic_density), 2)                AS avg_density,
    COUNT(*)                                        AS total_readings,

    -- Peak vs off-peak congestion lift (computed in outer query)
    ROUND(AVG(f.congestion_score), 2)               AS congestion_score_avg

FROM traffic_catalog.gold.fact_traffic_events     f
JOIN traffic_catalog.gold.dim_sensor              ds ON f.sensor_id = ds.sensor_id
JOIN traffic_catalog.gold.dim_time                dt ON f.time_key  = dt.time_key

GROUP BY
    f.sensor_id, ds.location_id,
    dt.is_peak_hour, dt.time_of_day
""")
views_created.append("vw_peak_vs_offpeak_analysis")
print("  ✓ vw_peak_vs_offpeak_analysis")

# ── View 3: daily_traffic_trend ──────────────────────────
# Answers: How does traffic evolve day by day across the network?
# Business use: detect deteriorating trends, seasonal patterns

spark.sql("""
CREATE OR REPLACE VIEW traffic_catalog.gold.vw_daily_traffic_trend AS
SELECT
    dd.full_date                                    AS event_date,
    dd.year,
    dd.month,
    dd.month_name,
    dd.weekday_name,
    dd.day_of_week,
    dd.is_weekend,
    dd.week_of_year,

    -- Daily network-wide averages
    ROUND(AVG(f.congestion_score), 2)               AS avg_congestion_score,
    ROUND(AVG(f.avg_speed), 2)                      AS avg_speed_kmh,
    ROUND(AVG(f.occupancy_rate), 4)                 AS avg_occupancy_rate,
    SUM(f.vehicle_count)                            AS total_vehicles,

    -- Sensors in critical congestion
    COUNT(CASE WHEN f.congestion_score >= 75 THEN 1 END)  AS critical_sensor_hours,
    COUNT(CASE WHEN f.congestion_score >= 50
               AND f.congestion_score < 75 THEN 1 END)    AS high_sensor_hours,

    -- Network health score (100 = perfect, 0 = fully congested)
    ROUND(100 - AVG(f.congestion_score), 2)         AS network_health_score,

    COUNT(DISTINCT f.sensor_id)                     AS active_sensors,
    COUNT(*)                                        AS total_readings

FROM traffic_catalog.gold.fact_traffic_events     f
JOIN traffic_catalog.gold.dim_date                dd ON f.date_key = dd.date_key

GROUP BY
    dd.full_date, dd.year, dd.month, dd.month_name,
    dd.weekday_name, dd.day_of_week, dd.is_weekend, dd.week_of_year

ORDER BY dd.full_date
""")
views_created.append("vw_daily_traffic_trend")
print("  ✓ vw_daily_traffic_trend")

# ── View 4: sensor_congestion_ranking ────────────────────
# Answers: Which are the top 10 worst sensors in the network?
# Business use: prioritise infrastructure investment

spark.sql("""
CREATE OR REPLACE VIEW traffic_catalog.gold.vw_sensor_congestion_ranking AS
SELECT
    f.sensor_id,
    ds.location_id,
    ds.lane_id,
    ds.sensor_type,

    ROUND(AVG(f.congestion_score), 2)               AS avg_congestion_score,
    ROUND(MAX(f.congestion_score), 2)               AS max_congestion_score,
    ROUND(AVG(f.avg_speed), 2)                      AS avg_speed_kmh,
    ROUND(AVG(f.occupancy_rate), 4)                 AS avg_occupancy_rate,
    SUM(f.vehicle_count)                            AS total_vehicles,

    -- % of time in critical congestion
    ROUND(
        COUNT(CASE WHEN f.congestion_score >= 75 THEN 1 END) * 100.0 / COUNT(*), 2
    )                                               AS pct_time_critical,

    -- % of time in high or critical congestion
    ROUND(
        COUNT(CASE WHEN f.congestion_score >= 50 THEN 1 END) * 100.0 / COUNT(*), 2
    )                                               AS pct_time_high_or_critical,

    COUNT(*)                                        AS total_readings,

    -- Rank by average congestion (1 = worst)
    RANK() OVER (ORDER BY AVG(f.congestion_score) DESC) AS congestion_rank

FROM traffic_catalog.gold.fact_traffic_events     f
JOIN traffic_catalog.gold.dim_sensor              ds ON f.sensor_id = ds.sensor_id

GROUP BY
    f.sensor_id, ds.location_id, ds.lane_id, ds.sensor_type
""")
views_created.append("vw_sensor_congestion_ranking")
print("  ✓ vw_sensor_congestion_ranking")

# ── View 5: weekend_vs_weekday_comparison ────────────────
# Answers: How different is weekend traffic vs weekday?
# Business use: adjust signal timing schedules by day type

spark.sql("""
CREATE OR REPLACE VIEW traffic_catalog.gold.vw_weekend_vs_weekday_comparison AS
SELECT
    dd.is_weekend,
    CASE WHEN dd.is_weekend THEN 'Weekend' ELSE 'Weekday' END AS day_type,
    dt.hour                                         AS event_hour,
    dt.time_of_day,

    ROUND(AVG(f.congestion_score), 2)               AS avg_congestion_score,
    ROUND(AVG(f.avg_speed), 2)                      AS avg_speed_kmh,
    ROUND(AVG(f.occupancy_rate), 4)                 AS avg_occupancy_rate,
    ROUND(AVG(f.traffic_density), 2)                AS avg_traffic_density,
    SUM(f.vehicle_count)                            AS total_vehicles,
    COUNT(*)                                        AS total_readings

FROM traffic_catalog.gold.fact_traffic_events     f
JOIN traffic_catalog.gold.dim_date                dd ON f.date_key = dd.date_key
JOIN traffic_catalog.gold.dim_time                dt ON f.time_key = dt.time_key

GROUP BY
    dd.is_weekend, dt.hour, dt.time_of_day

ORDER BY
    dd.is_weekend, dt.hour
""")
views_created.append("vw_weekend_vs_weekday_comparison")
print("  ✓ vw_weekend_vs_weekday_comparison")

# =========================================================
# CATEGORY 2 — INCIDENTS
# =========================================================

print("\n" + "=" * 50)
print("CATEGORY 2 — INCIDENTS")
print("=" * 50)

# ── View 6: daily_incident_report ────────────────────────
# Answers: How many incidents per day, by type and severity?
# Business use: daily operational reporting

spark.sql("""
CREATE OR REPLACE VIEW traffic_catalog.gold.vw_daily_incident_report AS
SELECT
    dd.full_date                                    AS event_date,
    dd.weekday_name,
    dd.is_weekend,
    dd.month_name,
    di.incident_type,
    di.incident_severity,
    di.severity_rank,

    COUNT(f.incident_id)                            AS incident_count,
    ROUND(AVG(f.incident_duration), 2)              AS avg_duration_mins,
    ROUND(MAX(f.incident_duration), 2)              AS max_duration_mins,
    ROUND(AVG(f.response_time), 2)                  AS avg_response_time_mins,
    ROUND(MIN(f.response_time), 2)                  AS min_response_time_mins,
    SUM(f.vehicles_affected)                        AS total_vehicles_affected,
    COUNT(CASE WHEN f.lane_blocked_flag = TRUE THEN 1 END) AS lane_blocked_count,

    -- % incidents where lane was blocked
    ROUND(
        COUNT(CASE WHEN f.lane_blocked_flag = TRUE THEN 1 END) * 100.0
        / NULLIF(COUNT(f.incident_id), 0), 2
    )                                               AS pct_lane_blocked

FROM traffic_catalog.gold.fact_traffic_events     f
JOIN traffic_catalog.gold.dim_date                dd ON f.date_key      = dd.date_key
JOIN traffic_catalog.gold.dim_incident            di ON f.incident_key  = di.incident_key

WHERE f.has_incident = TRUE

GROUP BY
    dd.full_date, dd.weekday_name, dd.is_weekend,
    dd.month_name, di.incident_type,
    di.incident_severity, di.severity_rank

ORDER BY dd.full_date, di.severity_rank DESC
""")
views_created.append("vw_daily_incident_report")
print("  ✓ vw_daily_incident_report")

# ── View 7: incident_severity_heatmap ────────────────────
# Answers: When and where do HIGH severity incidents cluster?
# Business use: emergency resource planning

spark.sql("""
CREATE OR REPLACE VIEW traffic_catalog.gold.vw_incident_severity_heatmap AS
SELECT
    f.sensor_id,
    ds.location_id,
    dt.hour                                         AS event_hour,
    dt.time_of_day,
    dt.is_peak_hour,
    dd.weekday_name,
    dd.is_weekend,
    di.incident_type,
    di.incident_severity,
    di.severity_rank,

    COUNT(f.incident_id)                            AS incident_count,
    ROUND(AVG(f.incident_duration), 2)              AS avg_duration_mins,
    ROUND(AVG(f.response_time), 2)                  AS avg_response_time,
    SUM(f.vehicles_affected)                        AS total_vehicles_affected,

    -- Congestion context when incident happened
    ROUND(AVG(f.congestion_score), 2)               AS avg_congestion_at_incident,
    ROUND(AVG(f.avg_speed), 2)                      AS avg_speed_at_incident

FROM traffic_catalog.gold.fact_traffic_events     f
JOIN traffic_catalog.gold.dim_sensor              ds ON f.sensor_id     = ds.sensor_id
JOIN traffic_catalog.gold.dim_time                dt ON f.time_key      = dt.time_key
JOIN traffic_catalog.gold.dim_date                dd ON f.date_key      = dd.date_key
JOIN traffic_catalog.gold.dim_incident            di ON f.incident_key  = di.incident_key

WHERE f.has_incident = TRUE

GROUP BY
    f.sensor_id, ds.location_id,
    dt.hour, dt.time_of_day, dt.is_peak_hour,
    dd.weekday_name, dd.is_weekend,
    di.incident_type, di.incident_severity, di.severity_rank
""")
views_created.append("vw_incident_severity_heatmap")
print("  ✓ vw_incident_severity_heatmap")

# ── View 8: incident_response_analysis ───────────────────
# Answers: Which incident types take longest to respond to?
# Business use: optimise emergency response allocation

spark.sql("""
CREATE OR REPLACE VIEW traffic_catalog.gold.vw_incident_response_analysis AS
SELECT
    di.incident_type,
    di.incident_severity,
    di.severity_rank,

    COUNT(f.incident_id)                            AS total_incidents,
    ROUND(AVG(f.response_time), 2)                  AS avg_response_time_mins,
    ROUND(MIN(f.response_time), 2)                  AS min_response_time_mins,
    ROUND(MAX(f.response_time), 2)                  AS max_response_time_mins,
    ROUND(STDDEV(f.response_time), 2)               AS stddev_response_time,

    ROUND(AVG(f.incident_duration), 2)              AS avg_incident_duration_mins,
    ROUND(MAX(f.incident_duration), 2)              AS max_incident_duration_mins,

    SUM(f.vehicles_affected)                        AS total_vehicles_affected,
    ROUND(AVG(f.vehicles_affected), 1)              AS avg_vehicles_affected,

    -- % cleared within 30 mins
    ROUND(
        COUNT(CASE WHEN f.response_time <= 30 THEN 1 END) * 100.0
        / NULLIF(COUNT(f.incident_id), 0), 2
    )                                               AS pct_responded_within_30_mins,

    -- % resolved within 60 mins
    ROUND(
        COUNT(CASE WHEN f.incident_duration <= 60 THEN 1 END) * 100.0
        / NULLIF(COUNT(f.incident_id), 0), 2
    )                                               AS pct_resolved_within_60_mins

FROM traffic_catalog.gold.fact_traffic_events     f
JOIN traffic_catalog.gold.dim_incident            di ON f.incident_key = di.incident_key

WHERE f.has_incident = TRUE

GROUP BY
    di.incident_type, di.incident_severity, di.severity_rank

ORDER BY di.severity_rank DESC, avg_response_time_mins DESC
""")
views_created.append("vw_incident_response_analysis")
print("  ✓ vw_incident_response_analysis")

# ── View 9: incident_peak_pattern ────────────────────────
# Answers: Do incidents cluster at specific hours?
# Business use: pre-position emergency teams before peak hours

spark.sql("""
CREATE OR REPLACE VIEW traffic_catalog.gold.vw_incident_peak_pattern AS
SELECT
    dt.hour                                         AS event_hour,
    dt.time_of_day,
    dt.is_peak_hour,
    dd.weekday_name,
    di.incident_type,
    di.incident_severity,

    COUNT(f.incident_id)                            AS incident_count,
    ROUND(AVG(f.congestion_score), 2)               AS avg_congestion_at_incident,
    ROUND(AVG(f.response_time), 2)                  AS avg_response_time,
    SUM(f.vehicles_affected)                        AS total_vehicles_affected,

    -- Hour share of all incidents
    ROUND(
        COUNT(f.incident_id) * 100.0
        / SUM(COUNT(f.incident_id)) OVER (PARTITION BY di.incident_type), 2
    )                                               AS pct_of_type_incidents

FROM traffic_catalog.gold.fact_traffic_events     f
JOIN traffic_catalog.gold.dim_time                dt ON f.time_key     = dt.time_key
JOIN traffic_catalog.gold.dim_date                dd ON f.date_key     = dd.date_key
JOIN traffic_catalog.gold.dim_incident            di ON f.incident_key = di.incident_key

WHERE f.has_incident = TRUE

GROUP BY
    dt.hour, dt.time_of_day, dt.is_peak_hour,
    dd.weekday_name, di.incident_type, di.incident_severity

ORDER BY dt.hour
""")
views_created.append("vw_incident_peak_pattern")
print("  ✓ vw_incident_peak_pattern")

# ── View 10: lane_blockage_impact ────────────────────────
# Answers: How much does a blocked lane increase congestion?
# Business use: quantify cost of incidents on traffic flow

spark.sql("""
CREATE OR REPLACE VIEW traffic_catalog.gold.vw_lane_blockage_impact AS
SELECT
    f.lane_blocked_flag,
    CASE WHEN f.lane_blocked_flag = TRUE THEN 'Lane Blocked' ELSE 'Lane Open' END
                                                    AS blockage_status,
    di.incident_type,
    di.incident_severity,

    COUNT(f.incident_id)                            AS incident_count,
    ROUND(AVG(f.congestion_score), 2)               AS avg_congestion_score,
    ROUND(AVG(f.avg_speed), 2)                      AS avg_speed_kmh,
    ROUND(AVG(f.occupancy_rate), 4)                 AS avg_occupancy_rate,
    ROUND(AVG(f.incident_duration), 2)              AS avg_incident_duration,
    ROUND(AVG(f.response_time), 2)                  AS avg_response_time,
    SUM(f.vehicles_affected)                        AS total_vehicles_affected

FROM traffic_catalog.gold.fact_traffic_events     f
JOIN traffic_catalog.gold.dim_incident            di ON f.incident_key = di.incident_key

WHERE f.has_incident = TRUE
  AND f.lane_blocked_flag IS NOT NULL

GROUP BY
    f.lane_blocked_flag, di.incident_type, di.incident_severity

ORDER BY f.lane_blocked_flag DESC, avg_congestion_score DESC
""")
views_created.append("vw_lane_blockage_impact")
print("  ✓ vw_lane_blockage_impact")

# =========================================================
# CATEGORY 3 — SPEED
# =========================================================

print("\n" + "=" * 50)
print("CATEGORY 3 — SPEED")
print("=" * 50)

# ── View 11: speed_violation_trends ──────────────────────
# Answers: Are violations increasing or decreasing over time?
# Business use: measure effectiveness of enforcement campaigns

spark.sql("""
CREATE OR REPLACE VIEW traffic_catalog.gold.vw_speed_violation_trends AS
SELECT
    dd.full_date                                    AS event_date,
    dd.weekday_name,
    dd.is_weekend,
    dd.month_name,
    dt.time_of_day,
    dt.is_peak_hour,

    COUNT(*)                                        AS total_speed_readings,
    COUNT(CASE WHEN f.speed_violation_flag = TRUE THEN 1 END)
                                                    AS total_violations,
    ROUND(
        COUNT(CASE WHEN f.speed_violation_flag = TRUE THEN 1 END) * 100.0
        / NULLIF(COUNT(*), 0), 2
    )                                               AS violation_rate_pct,

    ROUND(AVG(f.avg_speed), 2)                      AS avg_speed_kmh,
    ROUND(AVG(CASE WHEN f.speed_violation_flag = TRUE
                   THEN f.speed_excess END), 2)     AS avg_speed_excess_kmh,
    ROUND(MAX(f.speed_excess), 2)                   AS max_speed_excess_kmh,
    ROUND(AVG(f.speed_limit), 0)                    AS avg_speed_limit,
    SUM(f.speed_violation_count)                    AS total_violation_events

FROM traffic_catalog.gold.fact_traffic_events     f
JOIN traffic_catalog.gold.dim_date                dd ON f.date_key = dd.date_key
JOIN traffic_catalog.gold.dim_time                dt ON f.time_key = dt.time_key

WHERE f.speed_limit IS NOT NULL

GROUP BY
    dd.full_date, dd.weekday_name, dd.is_weekend,
    dd.month_name, dt.time_of_day, dt.is_peak_hour

ORDER BY dd.full_date
""")
views_created.append("vw_speed_violation_trends")
print("  ✓ vw_speed_violation_trends")

# ── View 12: top_violation_sensors ───────────────────────
# Answers: Which roads / sensors have the worst speeding problem?
# Business use: target enforcement resources and speed cameras

spark.sql("""
CREATE OR REPLACE VIEW traffic_catalog.gold.vw_top_violation_sensors AS
SELECT
    f.sensor_id,
    ds.location_id,
    ds.lane_id,

    COUNT(*)                                        AS total_readings,
    COUNT(CASE WHEN f.speed_violation_flag = TRUE THEN 1 END)
                                                    AS total_violations,
    ROUND(
        COUNT(CASE WHEN f.speed_violation_flag = TRUE THEN 1 END) * 100.0
        / NULLIF(COUNT(*), 0), 2
    )                                               AS violation_rate_pct,

    ROUND(AVG(f.avg_speed), 2)                      AS avg_speed_kmh,
    ROUND(AVG(f.speed_limit), 0)                    AS avg_speed_limit,
    ROUND(AVG(CASE WHEN f.speed_violation_flag = TRUE
                   THEN f.speed_excess END), 2)     AS avg_excess_when_violating,
    ROUND(MAX(f.speed_excess), 2)                   AS max_speed_excess,
    SUM(f.speed_violation_count)                    AS total_violation_events,

    -- Compliance rate (inverse of violation rate)
    ROUND(
        COUNT(CASE WHEN f.speed_violation_flag = FALSE THEN 1 END) * 100.0
        / NULLIF(COUNT(*), 0), 2
    )                                               AS compliance_rate_pct,

    RANK() OVER (ORDER BY
        COUNT(CASE WHEN f.speed_violation_flag = TRUE THEN 1 END) * 100.0
        / NULLIF(COUNT(*), 0) DESC
    )                                               AS violation_rank

FROM traffic_catalog.gold.fact_traffic_events     f
JOIN traffic_catalog.gold.dim_sensor              ds ON f.sensor_id = ds.sensor_id

WHERE f.speed_limit IS NOT NULL

GROUP BY
    f.sensor_id, ds.location_id, ds.lane_id
""")
views_created.append("vw_top_violation_sensors")
print("  ✓ vw_top_violation_sensors")

# ── View 13: speed_by_time_of_day ────────────────────────
# Answers: At what time of day do speeds drop most — or peak?
# Business use: correlate speed with congestion patterns

spark.sql("""
CREATE OR REPLACE VIEW traffic_catalog.gold.vw_speed_by_time_of_day AS
SELECT
    dt.hour                                         AS event_hour,
    dt.time_of_day,
    dt.is_peak_hour,
    dt.shift,
    dd.weekday_name,
    dd.is_weekend,

    ROUND(AVG(f.avg_speed), 2)                      AS avg_speed_kmh,
    ROUND(MIN(f.avg_speed), 2)                      AS min_speed_kmh,
    ROUND(MAX(f.avg_speed), 2)                      AS max_speed_kmh,
    ROUND(STDDEV(f.avg_speed), 2)                   AS stddev_speed,
    ROUND(AVG(f.speed_variance), 2)                 AS avg_speed_variance,
    ROUND(AVG(f.speed_limit), 0)                    AS avg_speed_limit,

    -- Speed index (ratio of actual to limit — > 1 means speeding on average)
    ROUND(AVG(f.speed_index), 4)                    AS avg_speed_index,

    COUNT(CASE WHEN f.speed_violation_flag = TRUE THEN 1 END)
                                                    AS violation_count,
    ROUND(
        COUNT(CASE WHEN f.speed_violation_flag = TRUE THEN 1 END) * 100.0
        / NULLIF(COUNT(*), 0), 2
    )                                               AS violation_rate_pct,
    COUNT(*)                                        AS total_readings

FROM traffic_catalog.gold.fact_traffic_events     f
JOIN traffic_catalog.gold.dim_time                dt ON f.time_key = dt.time_key
JOIN traffic_catalog.gold.dim_date                dd ON f.date_key = dd.date_key

WHERE f.speed_limit IS NOT NULL

GROUP BY
    dt.hour, dt.time_of_day, dt.is_peak_hour, dt.shift,
    dd.weekday_name, dd.is_weekend

ORDER BY dt.hour
""")
views_created.append("vw_speed_by_time_of_day")
print("  ✓ vw_speed_by_time_of_day")

# ── View 14: speed_compliance_rate ───────────────────────
# Answers: What is the overall compliance rate across the network?
# Business use: headline KPI for traffic authority reports

spark.sql("""
CREATE OR REPLACE VIEW traffic_catalog.gold.vw_speed_compliance_rate AS
SELECT
    dd.full_date                                    AS event_date,
    dd.month_name,
    dd.weekday_name,
    dd.is_weekend,
    f.sensor_id,
    ds.location_id,

    ROUND(AVG(f.speed_limit), 0)                    AS speed_limit,
    ROUND(AVG(f.avg_speed), 2)                      AS avg_speed_kmh,
    ROUND(AVG(f.sensor_avg_speed), 2)               AS sensor_avg_speed_kmh,

    COUNT(*)                                        AS total_readings,
    COUNT(CASE WHEN f.speed_violation_flag = FALSE THEN 1 END)
                                                    AS compliant_readings,
    COUNT(CASE WHEN f.speed_violation_flag = TRUE THEN 1 END)
                                                    AS violation_readings,

    ROUND(
        COUNT(CASE WHEN f.speed_violation_flag = FALSE THEN 1 END) * 100.0
        / NULLIF(COUNT(*), 0), 2
    )                                               AS compliance_rate_pct,

    ROUND(AVG(f.speed_index), 4)                    AS avg_speed_index,

    -- Compliance grade
    CASE
        WHEN COUNT(CASE WHEN f.speed_violation_flag = FALSE THEN 1 END) * 100.0
             / NULLIF(COUNT(*), 0) >= 90 THEN 'A — Excellent'
        WHEN COUNT(CASE WHEN f.speed_violation_flag = FALSE THEN 1 END) * 100.0
             / NULLIF(COUNT(*), 0) >= 75 THEN 'B — Good'
        WHEN COUNT(CASE WHEN f.speed_violation_flag = FALSE THEN 1 END) * 100.0
             / NULLIF(COUNT(*), 0) >= 60 THEN 'C — Average'
        ELSE 'D — Poor'
    END                                             AS compliance_grade

FROM traffic_catalog.gold.fact_traffic_events     f
JOIN traffic_catalog.gold.dim_date                dd ON f.date_key  = dd.date_key
JOIN traffic_catalog.gold.dim_sensor              ds ON f.sensor_id = ds.sensor_id

WHERE f.speed_limit IS NOT NULL

GROUP BY
    dd.full_date, dd.month_name, dd.weekday_name,
    dd.is_weekend, f.sensor_id, ds.location_id

ORDER BY dd.full_date, compliance_rate_pct ASC
""")
views_created.append("vw_speed_compliance_rate")
print("  ✓ vw_speed_compliance_rate")

# =========================================================
# VALIDATION — sample each view
# =========================================================

print("\n" + "=" * 50)
print("VIEW VALIDATION")
print("=" * 50)

all_ok = True
for view in views_created:
    try:
        df    = spark.table(f"traffic_catalog.gold.{view}")
        rows  = df.count()
        cols  = builtins.len(df.columns)
        print(f"  ✓ {view:<45} {rows:>6} rows | {cols} cols")
    except Exception as e:
        print(f"  ✗ {view} — FAILED: {e}")
        all_ok = False

print("\n" + "=" * 50)
print(f"Total views created : {builtins.len(views_created)}")
print(f"Status              : {'ALL PASSED ✓' if all_ok else 'SOME FAILED ✗'}")
print("=" * 50)

print("""
Sample queries to explore insights:

-- Top 10 most congested sensors
SELECT sensor_id, location_id, avg_congestion_score, pct_time_critical
FROM traffic_catalog.gold.vw_sensor_congestion_ranking
ORDER BY congestion_rank LIMIT 10;

-- Peak vs off-peak congestion lift
SELECT time_of_day, is_peak_hour, AVG(avg_congestion_score) AS score
FROM traffic_catalog.gold.vw_peak_vs_offpeak_analysis
GROUP BY time_of_day, is_peak_hour ORDER BY score DESC;

-- Worst incident response times
SELECT incident_type, incident_severity, avg_response_time_mins, pct_responded_within_30_mins
FROM traffic_catalog.gold.vw_incident_response_analysis
ORDER BY avg_response_time_mins DESC;

-- Sensors with worst speed compliance
SELECT sensor_id, location_id, violation_rate_pct, compliance_grade
FROM traffic_catalog.gold.vw_speed_compliance_rate
ORDER BY violation_rate_pct DESC LIMIT 10;

-- Daily network health trend
SELECT event_date, network_health_score, avg_congestion_score, total_vehicles
FROM traffic_catalog.gold.vw_daily_traffic_trend
ORDER BY event_date;
""")


# =========================================================
# GOLD LAYER — AGGREGATION VIEWS (REMAINING 3 CATEGORIES)
# =========================================================
# Category 4: Signal Performance (4 views)
# Category 5: Vehicle Mix        (4 views)
# Category 6: Business KPIs      (3 views)
# =========================================================

import builtins
from pyspark.sql import functions as F

views_created = []

# =========================================================
# CATEGORY 4 — SIGNAL PERFORMANCE
# =========================================================

print("=" * 50)
print("CATEGORY 4 — SIGNAL PERFORMANCE")
print("=" * 50)

# ── View 15: signal_efficiency_ranking ───────────────────
# Answers: Which signals are performing best/worst overall?
# Business use: prioritise signal retiming programmes

spark.sql("""
CREATE OR REPLACE VIEW traffic_catalog.gold.vw_signal_efficiency_ranking AS
SELECT
    f.signal_id,
    ds.location_id,
    ds.lane_id,

    COUNT(*)                                        AS total_readings,
    ROUND(AVG(f.signal_efficiency), 4)              AS avg_signal_efficiency,
    ROUND(MIN(f.signal_efficiency), 4)              AS min_signal_efficiency,
    ROUND(MAX(f.signal_efficiency), 4)              AS max_signal_efficiency,
    ROUND(STDDEV(f.signal_efficiency), 4)           AS stddev_efficiency,

    ROUND(AVG(f.green_ratio), 4)                    AS avg_green_ratio,
    ROUND(AVG(f.red_ratio), 4)                      AS avg_red_ratio,
    ROUND(AVG(f.signal_cycle_time), 2)              AS avg_cycle_time_secs,
    ROUND(AVG(f.signal_wait_time), 2)               AS avg_wait_time_secs,
    ROUND(AVG(f.queue_length), 2)                   AS avg_queue_length,

    -- Signal status distribution
    COUNT(CASE WHEN f.signal_efficiency >= 0.75 THEN 1 END)  AS good_readings,
    COUNT(CASE WHEN f.signal_efficiency >= 0.50
               AND f.signal_efficiency  < 0.75 THEN 1 END)  AS moderate_readings,
    COUNT(CASE WHEN f.signal_efficiency  < 0.50 THEN 1 END)  AS poor_readings,

    ROUND(
        COUNT(CASE WHEN f.signal_efficiency >= 0.75 THEN 1 END) * 100.0
        / NULLIF(COUNT(*), 0), 2
    )                                               AS pct_good,
    ROUND(
        COUNT(CASE WHEN f.signal_efficiency  < 0.50 THEN 1 END) * 100.0
        / NULLIF(COUNT(*), 0), 2
    )                                               AS pct_poor,

    -- Overall grade
    CASE
        WHEN AVG(f.signal_efficiency) >= 0.75 THEN 'GOOD'
        WHEN AVG(f.signal_efficiency) >= 0.50 THEN 'MODERATE'
        ELSE 'POOR'
    END                                             AS overall_grade,

    RANK() OVER (ORDER BY AVG(f.signal_efficiency) DESC) AS efficiency_rank

FROM traffic_catalog.gold.fact_traffic_events     f
JOIN traffic_catalog.gold.dim_sensor              ds ON f.sensor_id = ds.sensor_id

WHERE f.signal_id IS NOT NULL
  AND f.signal_efficiency IS NOT NULL

GROUP BY f.signal_id, ds.location_id, ds.lane_id
""")
views_created.append("vw_signal_efficiency_ranking")
print("  ✓ vw_signal_efficiency_ranking")

# ── View 16: worst_intersections ─────────────────────────
# Answers: Which intersections cause the most driver delay?
# Business use: target signal upgrade budget

spark.sql("""
CREATE OR REPLACE VIEW traffic_catalog.gold.vw_worst_intersections AS
SELECT
    f.signal_id,
    ds.location_id,
    ds.lane_id,
    dt.time_of_day,
    dt.is_peak_hour,

    COUNT(*)                                        AS total_readings,
    ROUND(AVG(f.signal_wait_time), 2)               AS avg_wait_time_secs,
    ROUND(MAX(f.signal_wait_time), 2)               AS max_wait_time_secs,
    ROUND(AVG(f.queue_length), 2)                   AS avg_queue_length,
    ROUND(MAX(f.queue_length), 2)                   AS max_queue_length,
    ROUND(AVG(f.signal_efficiency), 4)              AS avg_efficiency,
    ROUND(AVG(f.congestion_score), 2)               AS avg_congestion_score,
    ROUND(AVG(f.green_ratio), 4)                    AS avg_green_ratio,

    -- Total driver delay (wait_time × vehicles as proxy)
    ROUND(SUM(f.signal_wait_time * f.vehicle_count), 0)
                                                    AS total_driver_delay_proxy,

    RANK() OVER (
        PARTITION BY dt.is_peak_hour
        ORDER BY AVG(f.signal_wait_time) DESC
    )                                               AS wait_time_rank

FROM traffic_catalog.gold.fact_traffic_events     f
JOIN traffic_catalog.gold.dim_sensor              ds ON f.sensor_id = ds.sensor_id
JOIN traffic_catalog.gold.dim_time                dt ON f.time_key  = dt.time_key

WHERE f.signal_id IS NOT NULL
  AND f.signal_wait_time IS NOT NULL

GROUP BY
    f.signal_id, ds.location_id, ds.lane_id,
    dt.time_of_day, dt.is_peak_hour
""")
views_created.append("vw_worst_intersections")
print("  ✓ vw_worst_intersections")

# ── View 17: signal_wait_time_trend ──────────────────────
# Answers: Are signal wait times getting better or worse by day?
# Business use: measure impact of signal retiming interventions

spark.sql("""
CREATE OR REPLACE VIEW traffic_catalog.gold.vw_signal_wait_time_trend AS
SELECT
    dd.full_date                                    AS event_date,
    dd.weekday_name,
    dd.is_weekend,
    dt.time_of_day,
    dt.is_peak_hour,

    COUNT(DISTINCT f.signal_id)                     AS active_signals,
    ROUND(AVG(f.signal_wait_time), 2)               AS avg_wait_time_secs,
    ROUND(MAX(f.signal_wait_time), 2)               AS max_wait_time_secs,
    ROUND(AVG(f.queue_length), 2)                   AS avg_queue_length,
    ROUND(AVG(f.signal_efficiency), 4)              AS avg_signal_efficiency,
    ROUND(AVG(f.green_ratio), 4)                    AS avg_green_ratio,

    -- % signals classified as POOR
    ROUND(
        COUNT(CASE WHEN f.signal_efficiency < 0.50 THEN 1 END) * 100.0
        / NULLIF(COUNT(*), 0), 2
    )                                               AS pct_poor_signals,

    COUNT(*)                                        AS total_readings

FROM traffic_catalog.gold.fact_traffic_events     f
JOIN traffic_catalog.gold.dim_date                dd ON f.date_key = dd.date_key
JOIN traffic_catalog.gold.dim_time                dt ON f.time_key = dt.time_key

WHERE f.signal_id IS NOT NULL
  AND f.signal_wait_time IS NOT NULL

GROUP BY
    dd.full_date, dd.weekday_name,
    dd.is_weekend, dt.time_of_day, dt.is_peak_hour

ORDER BY dd.full_date
""")
views_created.append("vw_signal_wait_time_trend")
print("  ✓ vw_signal_wait_time_trend")

# ── View 18: green_ratio_analysis ────────────────────────
# Answers: Are signals giving enough green time relative to demand?
# Business use: optimise green phase allocation

spark.sql("""
CREATE OR REPLACE VIEW traffic_catalog.gold.vw_green_ratio_analysis AS
SELECT
    f.signal_id,
    ds.location_id,
    dt.hour                                         AS event_hour,
    dt.time_of_day,
    dt.is_peak_hour,

    ROUND(AVG(f.green_ratio), 4)                    AS avg_green_ratio,
    ROUND(AVG(f.red_ratio), 4)                      AS avg_red_ratio,
    ROUND(1 - AVG(f.green_ratio) - AVG(f.red_ratio), 4)
                                                    AS avg_other_phase_ratio,
    ROUND(AVG(f.signal_cycle_time), 2)              AS avg_cycle_time_secs,
    ROUND(AVG(f.green_time), 2)                     AS avg_green_time_secs,
    ROUND(AVG(f.red_time), 2)                       AS avg_red_time_secs,
    ROUND(AVG(f.signal_wait_time), 2)               AS avg_wait_time_secs,
    ROUND(AVG(f.vehicle_count), 1)                  AS avg_vehicle_count,
    ROUND(AVG(f.signal_efficiency), 4)              AS avg_efficiency,

    -- Is green time proportional to vehicle demand?
    CASE
        WHEN AVG(f.green_ratio) >= 0.50 THEN 'Green dominant — good for flow'
        WHEN AVG(f.green_ratio) >= 0.35 THEN 'Balanced — adequate'
        ELSE 'Red dominant — flow restricted'
    END                                             AS phase_assessment,

    COUNT(*)                                        AS total_readings

FROM traffic_catalog.gold.fact_traffic_events     f
JOIN traffic_catalog.gold.dim_sensor              ds ON f.sensor_id = ds.sensor_id
JOIN traffic_catalog.gold.dim_time                dt ON f.time_key  = dt.time_key

WHERE f.signal_id IS NOT NULL
  AND f.green_ratio IS NOT NULL
  AND f.red_ratio   IS NOT NULL

GROUP BY
    f.signal_id, ds.location_id,
    dt.hour, dt.time_of_day, dt.is_peak_hour

ORDER BY dt.hour
""")
views_created.append("vw_green_ratio_analysis")
print("  ✓ vw_green_ratio_analysis")

# =========================================================
# CATEGORY 5 — VEHICLE MIX
# =========================================================

print("\n" + "=" * 50)
print("CATEGORY 5 — VEHICLE MIX")
print("=" * 50)

# ── View 19: vehicle_type_distribution ───────────────────
# Answers: What is the mix of vehicle types by hour / location?
# Business use: road design, toll policy, emission planning

spark.sql("""
CREATE OR REPLACE VIEW traffic_catalog.gold.vw_vehicle_type_distribution AS
SELECT
    f.vehicle_type,
    dv.vehicle_category,
    dv.is_heavy_vehicle,
    dt.hour                                         AS event_hour,
    dt.time_of_day,
    dt.is_peak_hour,
    dd.weekday_name,
    dd.is_weekend,

    COUNT(*)                                        AS total_readings,
    SUM(f.vehicle_type_count)                       AS total_vehicles,
    ROUND(AVG(f.vehicle_type_count), 1)             AS avg_vehicles_per_reading,
    ROUND(AVG(f.vehicle_flow_rate), 2)              AS avg_flow_rate,
    ROUND(AVG(f.vehicle_density), 2)                AS avg_density,
    ROUND(AVG(f.vehicle_delay), 2)                  AS avg_delay_secs,
    ROUND(AVG(f.lane_utilization), 4)               AS avg_lane_utilization,
    ROUND(AVG(f.queue_length), 2)                   AS avg_queue_length,

    -- Share of total vehicle count (window function)
    ROUND(
        SUM(f.vehicle_type_count) * 100.0
        / SUM(SUM(f.vehicle_type_count)) OVER (
            PARTITION BY dt.hour, dd.is_weekend
        ), 2
    )                                               AS pct_share_of_hourly_traffic

FROM traffic_catalog.gold.fact_traffic_events     f
JOIN traffic_catalog.gold.dim_vehicle             dv ON f.vehicle_type = dv.vehicle_type
JOIN traffic_catalog.gold.dim_time                dt ON f.time_key     = dt.time_key
JOIN traffic_catalog.gold.dim_date                dd ON f.date_key     = dd.date_key

WHERE f.vehicle_type IS NOT NULL

GROUP BY
    f.vehicle_type, dv.vehicle_category, dv.is_heavy_vehicle,
    dt.hour, dt.time_of_day, dt.is_peak_hour,
    dd.weekday_name, dd.is_weekend

ORDER BY dt.hour, total_vehicles DESC
""")
views_created.append("vw_vehicle_type_distribution")
print("  ✓ vw_vehicle_type_distribution")

# ── View 20: heavy_vehicle_impact ────────────────────────
# Answers: Do heavy vehicles (trucks/buses) worsen congestion?
# Business use: justify heavy vehicle restrictions in peak hours

spark.sql("""
CREATE OR REPLACE VIEW traffic_catalog.gold.vw_heavy_vehicle_impact AS
SELECT
    dv.is_heavy_vehicle,
    CASE WHEN dv.is_heavy_vehicle THEN 'Heavy (TRUCK/BUS)'
         ELSE 'Light (CAR/BIKE)' END                AS vehicle_class,
    dt.time_of_day,
    dt.is_peak_hour,

    COUNT(*)                                        AS total_readings,
    SUM(f.vehicle_type_count)                       AS total_vehicles,
    ROUND(AVG(f.congestion_score), 2)               AS avg_congestion_score,
    ROUND(AVG(f.avg_speed), 2)                      AS avg_speed_kmh,
    ROUND(AVG(f.vehicle_delay), 2)                  AS avg_delay_secs,
    ROUND(AVG(f.queue_length), 2)                   AS avg_queue_length,
    ROUND(AVG(f.lane_utilization), 4)               AS avg_lane_utilization,
    ROUND(AVG(f.vehicle_density), 2)                AS avg_vehicle_density,
    ROUND(AVG(f.vehicle_flow_rate), 2)              AS avg_flow_rate,

    -- Incident co-occurrence rate
    ROUND(
        COUNT(CASE WHEN f.has_incident = TRUE THEN 1 END) * 100.0
        / NULLIF(COUNT(*), 0), 2
    )                                               AS incident_co_occurrence_pct

FROM traffic_catalog.gold.fact_traffic_events     f
JOIN traffic_catalog.gold.dim_vehicle             dv ON f.vehicle_type = dv.vehicle_type
JOIN traffic_catalog.gold.dim_time                dt ON f.time_key     = dt.time_key

WHERE f.vehicle_type IS NOT NULL

GROUP BY
    dv.is_heavy_vehicle, dt.time_of_day, dt.is_peak_hour

ORDER BY dt.is_peak_hour DESC, avg_congestion_score DESC
""")
views_created.append("vw_heavy_vehicle_impact")
print("  ✓ vw_heavy_vehicle_impact")

# ── View 21: vehicle_flow_by_hour ────────────────────────
# Answers: When does vehicle flow peak for each vehicle type?
# Business use: schedule deliveries, optimise bus timetables

spark.sql("""
CREATE OR REPLACE VIEW traffic_catalog.gold.vw_vehicle_flow_by_hour AS
SELECT
    f.vehicle_type,
    dv.vehicle_category,
    dv.is_heavy_vehicle,
    dt.hour                                         AS event_hour,
    dt.time_of_day,
    dt.is_peak_hour,

    ROUND(AVG(f.vehicle_flow_rate), 2)              AS avg_flow_rate,
    ROUND(MAX(f.vehicle_flow_rate), 2)              AS max_flow_rate,
    SUM(f.vehicle_type_count)                       AS total_vehicles,
    ROUND(AVG(f.vehicle_density), 2)                AS avg_density,
    ROUND(AVG(f.vehicle_delay), 2)                  AS avg_delay_secs,
    ROUND(AVG(f.lane_utilization), 4)               AS avg_lane_utilization,

    -- Peak hour flow index (relative to daily average)
    ROUND(
        AVG(f.vehicle_flow_rate)
        / NULLIF(AVG(AVG(f.vehicle_flow_rate)) OVER (
            PARTITION BY f.vehicle_type
        ), 0), 4
    )                                               AS flow_index_vs_daily_avg,

    COUNT(*)                                        AS total_readings

FROM traffic_catalog.gold.fact_traffic_events     f
JOIN traffic_catalog.gold.dim_vehicle             dv ON f.vehicle_type = dv.vehicle_type
JOIN traffic_catalog.gold.dim_time                dt ON f.time_key     = dt.time_key

WHERE f.vehicle_type IS NOT NULL

GROUP BY
    f.vehicle_type, dv.vehicle_category, dv.is_heavy_vehicle,
    dt.hour, dt.time_of_day, dt.is_peak_hour

ORDER BY f.vehicle_type, dt.hour
""")
views_created.append("vw_vehicle_flow_by_hour")
print("  ✓ vw_vehicle_flow_by_hour")

# ── View 22: queue_length_analysis ───────────────────────
# Answers: Where and when do the longest queues form?
# Business use: identify roads needing capacity expansion

spark.sql("""
CREATE OR REPLACE VIEW traffic_catalog.gold.vw_queue_length_analysis AS
SELECT
    f.sensor_id,
    ds.location_id,
    ds.lane_id,
    dt.time_of_day,
    dt.is_peak_hour,
    dd.weekday_name,
    dd.is_weekend,

    ROUND(AVG(f.queue_length), 2)                   AS avg_queue_length,
    ROUND(MAX(f.queue_length), 2)                   AS max_queue_length,
    ROUND(STDDEV(f.queue_length), 2)                AS stddev_queue_length,

    -- Queue severity bucketing
    COUNT(CASE WHEN f.queue_length < 5  THEN 1 END) AS short_queue_count,
    COUNT(CASE WHEN f.queue_length < 15
               AND f.queue_length >= 5  THEN 1 END) AS medium_queue_count,
    COUNT(CASE WHEN f.queue_length >= 15 THEN 1 END) AS long_queue_count,

    ROUND(AVG(f.congestion_score), 2)               AS avg_congestion_score,
    ROUND(AVG(f.vehicle_delay), 2)                  AS avg_vehicle_delay,
    ROUND(AVG(f.lane_utilization), 4)               AS avg_lane_utilization,

    -- Queue to congestion correlation indicator
    ROUND(AVG(f.queue_length) / NULLIF(AVG(f.congestion_score), 0), 4)
                                                    AS queue_per_congestion_unit,

    RANK() OVER (
        PARTITION BY dt.is_peak_hour
        ORDER BY AVG(f.queue_length) DESC
    )                                               AS queue_rank,

    COUNT(*)                                        AS total_readings

FROM traffic_catalog.gold.fact_traffic_events     f
JOIN traffic_catalog.gold.dim_sensor              ds ON f.sensor_id = ds.sensor_id
JOIN traffic_catalog.gold.dim_time                dt ON f.time_key  = dt.time_key
JOIN traffic_catalog.gold.dim_date                dd ON f.date_key  = dd.date_key

WHERE f.queue_length IS NOT NULL

GROUP BY
    f.sensor_id, ds.location_id, ds.lane_id,
    dt.time_of_day, dt.is_peak_hour,
    dd.weekday_name, dd.is_weekend
""")
views_created.append("vw_queue_length_analysis")
print("  ✓ vw_queue_length_analysis")

# =========================================================
# CATEGORY 6 — BUSINESS KPIs
# =========================================================

print("\n" + "=" * 50)
print("CATEGORY 6 — BUSINESS KPIs")
print("=" * 50)

# ── View 23: overall_traffic_health_score ────────────────
# Answers: What is the single headline KPI for the network today?
# Business use: C-suite / authority daily briefing dashboard

spark.sql("""
CREATE OR REPLACE VIEW traffic_catalog.gold.vw_overall_traffic_health AS
SELECT
    dd.full_date                                    AS event_date,
    dd.month_name,
    dd.weekday_name,
    dd.is_weekend,
    dd.week_of_year,

    -- Core KPIs
    ROUND(AVG(f.congestion_score), 2)               AS avg_congestion_score,
    ROUND(100 - AVG(f.congestion_score), 2)         AS network_health_score,
    ROUND(AVG(f.avg_speed), 2)                      AS avg_network_speed_kmh,
    ROUND(AVG(f.occupancy_rate) * 100, 2)           AS avg_occupancy_pct,
    SUM(f.vehicle_count)                            AS total_vehicles,

    -- Incident KPIs
    COUNT(CASE WHEN f.has_incident = TRUE THEN 1 END)
                                                    AS total_incidents,
    COUNT(CASE WHEN f.has_incident = TRUE
               AND f.is_severe     = TRUE THEN 1 END)
                                                    AS high_severity_incidents,
    ROUND(AVG(CASE WHEN f.has_incident = TRUE
                   THEN f.response_time END), 2)    AS avg_incident_response_mins,

    -- Speed KPIs
    COUNT(CASE WHEN f.has_speed_violation = TRUE THEN 1 END)
                                                    AS total_speed_violations,
    ROUND(
        COUNT(CASE WHEN f.has_speed_violation = TRUE THEN 1 END) * 100.0
        / NULLIF(COUNT(*), 0), 2
    )                                               AS violation_rate_pct,

    -- Signal KPIs
    ROUND(AVG(CASE WHEN f.signal_id IS NOT NULL
                   THEN f.signal_efficiency END), 4) AS avg_signal_efficiency,
    ROUND(AVG(CASE WHEN f.signal_id IS NOT NULL
                   THEN f.signal_wait_time END), 2)  AS avg_signal_wait_secs,

    -- Critical congestion exposure
    ROUND(
        COUNT(CASE WHEN f.congestion_score >= 75 THEN 1 END) * 100.0
        / NULLIF(COUNT(*), 0), 2
    )                                               AS pct_time_in_critical_congestion,

    COUNT(DISTINCT f.sensor_id)                     AS active_sensors,

    -- Overall traffic grade
    CASE
        WHEN AVG(f.congestion_score) <  25 THEN 'A — Excellent'
        WHEN AVG(f.congestion_score) <  50 THEN 'B — Good'
        WHEN AVG(f.congestion_score) <  75 THEN 'C — Moderate'
        ELSE 'D — Poor'
    END                                             AS daily_traffic_grade

FROM traffic_catalog.gold.fact_traffic_events     f
JOIN traffic_catalog.gold.dim_date                dd ON f.date_key = dd.date_key

GROUP BY
    dd.full_date, dd.month_name, dd.weekday_name,
    dd.is_weekend, dd.week_of_year

ORDER BY dd.full_date
""")
views_created.append("vw_overall_traffic_health")
print("  ✓ vw_overall_traffic_health")

# ── View 24: monthly_summary ─────────────────────────────
# Answers: How does each month compare across all KPIs?
# Business use: monthly management report, trend analysis

spark.sql("""
CREATE OR REPLACE VIEW traffic_catalog.gold.vw_monthly_summary AS
SELECT
    dd.year,
    dd.month,
    dd.month_name,

    -- Volume
    SUM(f.vehicle_count)                            AS total_vehicles,
    COUNT(DISTINCT dd.full_date)                    AS active_days,
    ROUND(SUM(f.vehicle_count) / NULLIF(COUNT(DISTINCT dd.full_date), 0), 0)
                                                    AS avg_daily_vehicles,

    -- Congestion
    ROUND(AVG(f.congestion_score), 2)               AS avg_congestion_score,
    ROUND(100 - AVG(f.congestion_score), 2)         AS avg_health_score,
    ROUND(AVG(f.avg_speed), 2)                      AS avg_speed_kmh,

    -- Incidents
    COUNT(CASE WHEN f.has_incident = TRUE THEN 1 END)
                                                    AS total_incidents,
    COUNT(CASE WHEN f.has_incident = TRUE
               AND f.is_severe     = TRUE THEN 1 END)
                                                    AS high_severity_incidents,
    ROUND(AVG(CASE WHEN f.has_incident = TRUE
                   THEN f.response_time END), 2)    AS avg_response_time_mins,

    -- Speed
    ROUND(
        COUNT(CASE WHEN f.has_speed_violation = TRUE THEN 1 END) * 100.0
        / NULLIF(COUNT(*), 0), 2
    )                                               AS violation_rate_pct,

    -- Signals
    ROUND(AVG(CASE WHEN f.signal_id IS NOT NULL
                   THEN f.signal_efficiency END), 4) AS avg_signal_efficiency,

    COUNT(*)                                        AS total_readings

FROM traffic_catalog.gold.fact_traffic_events     f
JOIN traffic_catalog.gold.dim_date                dd ON f.date_key = dd.date_key

GROUP BY dd.year, dd.month, dd.month_name
ORDER BY dd.year, dd.month
""")
views_created.append("vw_monthly_summary")
print("  ✓ vw_monthly_summary")

# ── View 25: sensor_performance_scorecard ────────────────
# Answers: How healthy is each sensor across ALL dimensions?
# Business use: sensor maintenance scheduling, network audit

spark.sql("""
CREATE OR REPLACE VIEW traffic_catalog.gold.vw_sensor_performance_scorecard AS
SELECT
    f.sensor_id,
    ds.location_id,
    ds.lane_id,
    ds.sensor_type,

    -- Traffic flow score
    ROUND(AVG(f.congestion_score), 2)               AS avg_congestion_score,
    ROUND(AVG(f.avg_speed), 2)                      AS avg_speed_kmh,
    ROUND(AVG(f.occupancy_rate), 4)                 AS avg_occupancy_rate,
    SUM(f.vehicle_count)                            AS total_vehicles,

    -- Incident score
    COUNT(CASE WHEN f.has_incident = TRUE THEN 1 END)
                                                    AS total_incidents,
    ROUND(
        COUNT(CASE WHEN f.has_incident = TRUE THEN 1 END) * 100.0
        / NULLIF(COUNT(*), 0), 2
    )                                               AS incident_rate_pct,
    ROUND(AVG(CASE WHEN f.has_incident = TRUE
                   THEN f.response_time END), 2)    AS avg_response_time,

    -- Speed score
    ROUND(
        COUNT(CASE WHEN f.has_speed_violation = TRUE THEN 1 END) * 100.0
        / NULLIF(COUNT(*), 0), 2
    )                                               AS violation_rate_pct,

    -- Signal score
    ROUND(AVG(CASE WHEN f.signal_id IS NOT NULL
                   THEN f.signal_efficiency END), 4) AS avg_signal_efficiency,
    ROUND(AVG(CASE WHEN f.signal_id IS NOT NULL
                   THEN f.signal_wait_time END), 2)  AS avg_signal_wait,

    -- Data quality
    ROUND(AVG(f.dq_score), 2)                       AS avg_dq_score,
    COUNT(CASE WHEN f.is_outlier = TRUE THEN 1 END) AS outlier_readings,
    COUNT(*)                                        AS total_readings,

    -- Composite performance score (0–100, higher = better)
    ROUND(
        (100 - AVG(f.congestion_score)) * 0.35        -- 35% weight: low congestion good
        + (COALESCE(AVG(CASE WHEN f.signal_id IS NOT NULL
                    THEN f.signal_efficiency END), 0.6) * 100) * 0.25  -- 25%: high efficiency good
        + (100 - (COUNT(CASE WHEN f.has_speed_violation = TRUE THEN 1 END) * 100.0
                  / NULLIF(COUNT(*), 0))) * 0.25       -- 25%: low violation rate good
        + (100 - (COUNT(CASE WHEN f.has_incident = TRUE THEN 1 END) * 100.0
                  / NULLIF(COUNT(*), 0))) * 0.15       -- 15%: low incident rate good
    , 2)                                            AS composite_performance_score,

    RANK() OVER (
        ORDER BY
        (100 - AVG(f.congestion_score)) * 0.35
        + (COALESCE(AVG(CASE WHEN f.signal_id IS NOT NULL
                    THEN f.signal_efficiency END), 0.6) * 100) * 0.25
        + (100 - (COUNT(CASE WHEN f.has_speed_violation = TRUE THEN 1 END) * 100.0
                  / NULLIF(COUNT(*), 0))) * 0.25
        + (100 - (COUNT(CASE WHEN f.has_incident = TRUE THEN 1 END) * 100.0
                  / NULLIF(COUNT(*), 0))) * 0.15
        DESC
    )                                               AS performance_rank

FROM traffic_catalog.gold.fact_traffic_events     f
JOIN traffic_catalog.gold.dim_sensor              ds ON f.sensor_id = ds.sensor_id

GROUP BY
    f.sensor_id, ds.location_id, ds.lane_id, ds.sensor_type
""")
views_created.append("vw_sensor_performance_scorecard")
print("  ✓ vw_sensor_performance_scorecard")

# =========================================================
# VALIDATION
# =========================================================

print("\n" + "=" * 50)
print("VIEW VALIDATION")
print("=" * 50)

all_ok = True
for view in views_created:
    try:
        df   = spark.table(f"traffic_catalog.gold.{view}")
        rows = df.count()
        cols = builtins.len(df.columns)
        print(f"  ✓ {view:<45} {rows:>6} rows | {cols} cols")
    except Exception as e:
        print(f"  ✗ {view} — FAILED: {e}")
        all_ok = False

print("\n" + "=" * 50)
print(f"Total views created : {builtins.len(views_created)}")
print(f"Status              : {'ALL PASSED ✓' if all_ok else 'SOME FAILED ✗'}")
print("=" * 50)

print("""
Business insight queries:

-- Overall daily traffic grade
SELECT event_date, daily_traffic_grade, network_health_score,
       total_incidents, violation_rate_pct
FROM traffic_catalog.gold.vw_overall_traffic_health
ORDER BY event_date;

-- Top 5 best and worst performing sensors
SELECT sensor_id, location_id, composite_performance_score, performance_rank
FROM traffic_catalog.gold.vw_sensor_performance_scorecard
ORDER BY performance_rank LIMIT 5;

-- Do heavy vehicles worsen congestion during peak hours?
SELECT vehicle_class, time_of_day, is_peak_hour,
       avg_congestion_score, avg_delay_secs, avg_queue_length
FROM traffic_catalog.gold.vw_heavy_vehicle_impact
ORDER BY is_peak_hour DESC, avg_congestion_score DESC;

-- Worst intersections during peak hours
SELECT signal_id, location_id, avg_wait_time_secs,
       avg_queue_length, avg_efficiency
FROM traffic_catalog.gold.vw_worst_intersections
WHERE is_peak_hour = true
ORDER BY wait_time_rank LIMIT 10;

-- Monthly KPI comparison
SELECT month_name, avg_health_score, total_incidents,
       violation_rate_pct, avg_signal_efficiency
FROM traffic_catalog.gold.vw_monthly_summary
ORDER BY month;
""")